In [2]:
import bnlearn as bn
import numpy as np
from causallearn.search.ConstraintBased.PC import pc

# load ALARM
alarm_model = bn.import_DAG('alarm')

# Generate sampling 5k from Alarm bayes network
df_alarm = bn.sampling(alarm_model, n = 5000, methodtype='bayes')
#print(df_alarm)

data = df_alarm.to_numpy()
print('data:', data)

cg = pc(data, indep_test='chisq', node_names=list(df_alarm.columns))

c:\Users\24280666\AppData\Local\miniconda3\envs\causal_env\lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


[bnlearn] >Import <alarm>
[bnlearn] >Loading bif file <c:\Users\24280666\AppData\Local\miniconda3\envs\causal_env\lib\site-packages\datazets\data\alarm.bif>


[bnlearn] >[CPD > Validate] >[Node ANAPHYLAXIS] >OK
[bnlearn] >[CPD > Validate] >[Node ARTCO2] >OK
[bnlearn] >[CPD > Validate] >[Node BP] >Table Error: Does not sum to 1 but is [[[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]]
[bnlearn] >[CPD > Validate] >[Node CATECHOL] >OK
[bnlearn] >[CPD > Validate] >[Node CO] >OK
[bnlearn] >[CPD > Validate] >[Node CVP] >OK
[bnlearn] >[CPD > Validate] >[Node DISCONNECT] >OK
[bnlearn] >[CPD > Validate] >[Node ERRCAUTER] >OK
[bnlearn] >[CPD > Validate] >[Node ERRLOWOUTPUT] >OK
[bnlearn] >[CPD > Validate] >[Node EXPCO2] >OK
[bnlearn] >[CPD > Validate] >[Node FIO2] >OK
[bnlearn] >[CPD > Validate] >[Node HISTORY] >OK
[bnlearn] >[CPD > Validate] >[Node HR] >OK
[bnlearn] >[CPD > Validate] >[Node HRBP] >OK
[bnlearn] >[CPD > Validate] >[Node HREKG] >Table Error: Does not sum to 1 but is [[[0.9999999 0.9999999 1.       ]
 [0.9999999 1.        1.       ]]]
[bnlearn] >[CPD > Validate] >[Node HRSAT] >Table Error: Does not sum to 1 but is [[[0.9999999 0.9999999 1.       ]
 

  0%|          | 0/37 [00:00<?, ?it/s]

In [3]:
# Build true graph

from causallearn.graph.GeneralGraph import GeneralGraph
from causallearn.graph.GraphNode import GraphNode
from causallearn.graph.Edge import Edge
from causallearn.graph.Endpoint import Endpoint

# adj matrix
adj_mat = alarm_model['adjmat']

# create node objects same name with node_names used in pc
node_names = list(df_alarm.columns)
nodes = [GraphNode(name) for name in node_names]
node_dict = {n.get_name(): n for n in nodes}
truth_graph = GeneralGraph(nodes)
print("Adj matrix:", adj_mat)

for source in adj_mat.index:
    for target in adj_mat.columns:
        if adj_mat.loc[source, target] == 1:
            edge = Edge(node_dict[source], node_dict[target], Endpoint.TAIL, Endpoint.ARROW)
            truth_graph.add_edge(edge)

Adj matrix: target        LVFAILURE  HISTORY  LVEDVOLUME    CVP   PCWP  HYPOVOLEMIA  \
source                                                                    
LVFAILURE         False     True        True  False  False        False   
HISTORY           False    False       False  False  False        False   
LVEDVOLUME        False    False       False   True   True        False   
CVP               False    False       False  False  False        False   
PCWP              False    False       False  False  False        False   
HYPOVOLEMIA       False    False        True  False  False        False   
STROKEVOLUME      False    False       False  False  False        False   
ERRLOWOUTPUT      False    False       False  False  False        False   
HRBP              False    False       False  False  False        False   
HR                False    False       False  False  False        False   
ERRCAUTER         False    False       False  False  False        False   
HREKG        

In [4]:
# calculate SHD
# print(set(node_names) == set(adj_mat.index))

from causallearn.graph.SHD import SHD
shd = SHD(truth_graph, cg.G).get_shd()
print('SHD: ', shd)

### print
#true
print(alarm_model.keys())
print(type(alarm_model['model']))
true_model = alarm_model['model']

print("Number of nodes:", len(true_model.nodes()))

#for node in true_model.nodes():
#    print(node)
true_edges = list(true_model.edges())
print("Number of true edge:", len(true_edges))
for edge in true_edges:
    print(edge)

# learn
print("Number of learn edges:", len(cg.G.get_graph_edges()))
for edge in cg.G.get_graph_edges():
    print(edge)

SHD:  8
dict_keys(['model', 'adjmat'])
<class 'pgmpy.models.BayesianNetwork.BayesianNetwork'>
Number of nodes: 37
Number of true edge: 46
('LVFAILURE', 'HISTORY')
('LVFAILURE', 'LVEDVOLUME')
('LVFAILURE', 'STROKEVOLUME')
('LVEDVOLUME', 'CVP')
('LVEDVOLUME', 'PCWP')
('HYPOVOLEMIA', 'LVEDVOLUME')
('HYPOVOLEMIA', 'STROKEVOLUME')
('STROKEVOLUME', 'CO')
('ERRLOWOUTPUT', 'HRBP')
('HR', 'HRBP')
('HR', 'HREKG')
('HR', 'HRSAT')
('HR', 'CO')
('ERRCAUTER', 'HREKG')
('ERRCAUTER', 'HRSAT')
('ANAPHYLAXIS', 'TPR')
('TPR', 'CATECHOL')
('TPR', 'BP')
('ARTCO2', 'EXPCO2')
('ARTCO2', 'CATECHOL')
('VENTLUNG', 'EXPCO2')
('VENTLUNG', 'MINVOL')
('VENTLUNG', 'VENTALV')
('INTUBATION', 'MINVOL')
('INTUBATION', 'SHUNT')
('INTUBATION', 'PRESS')
('INTUBATION', 'VENTLUNG')
('INTUBATION', 'VENTALV')
('FIO2', 'PVSAT')
('PVSAT', 'SAO2')
('VENTALV', 'PVSAT')
('VENTALV', 'ARTCO2')
('SAO2', 'CATECHOL')
('SHUNT', 'SAO2')
('PULMEMBOLUS', 'PAP')
('PULMEMBOLUS', 'SHUNT')
('KINKEDTUBE', 'PRESS')
('KINKEDTUBE', 'VENTLUNG')
('VE